In [10]:
# !pip install transformers gradio scikit-learn -q

import pandas as pd
import numpy as np

from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import gradio as gr

In [11]:
data = [
    {"ticket": "Internet is slow", "label": "Network Issue"},
    {"ticket": "WiFi disconnects frequently", "label": "Network Issue"},
    {"ticket": "Server not responding", "label": "Network Issue"},

    {"ticket": "Cannot login to account", "label": "Authentication"},
    {"ticket": "Forgot my password", "label": "Authentication"},
    {"ticket": "Login page not working", "label": "Authentication"},

    {"ticket": "Payment failed", "label": "Billing"},
    {"ticket": "Charged twice", "label": "Billing"},
    {"ticket": "Refund not received", "label": "Billing"},

    {"ticket": "App crashes frequently", "label": "Technical Issue"},
    {"ticket": "App freezes on startup", "label": "Technical Issue"},
    {"ticket": "Bug in latest update", "label": "Technical Issue"},
]

df = pd.DataFrame(data)
labels = df["label"].unique().tolist()
df.head()

,ticket,label
0,Internet is slow,Network Issue
1,WiFi disconnects frequently,Network Issue
2,Server not responding,Network Issue
3,Cannot login to account,Authentication
4,Forgot my password,Authentication


In [12]:
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

def zero_shot_predict(text):
    result = zero_shot(text, labels)
    return result["labels"][:3]

df["zero_shot_tags"] = df["ticket"].apply(zero_shot_predict)
df["zero_top1"] = df["zero_shot_tags"].apply(lambda x: x[0])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [31]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
few_shot_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def few_shot_predict(ticket):
    prompt = f"""
Classify the support ticket into one category:
Network Issue, Authentication, Billing, Technical Issue

Ticket: {ticket}
Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = few_shot_model.generate(**inputs, max_new_tokens=20)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Flan-T5 is an instruction-tuned model; it should directly generate the answer.
    # The output should be the classified label.
    return generated_text.strip()

df["few_shot_pred"] = df["ticket"].apply(few_shot_predict)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [15]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["ticket"])
y = df["label"]

ml_model = LogisticRegression()
ml_model.fit(X, y)

def ml_predict(text):
    vec = vectorizer.transform([text])
    return ml_model.predict(vec)[0]

df["ml_pred"] = df["ticket"].apply(ml_predict)

In [16]:
# Top-1 Accuracy
zero_acc = accuracy_score(df["label"], df["zero_top1"])
ml_acc = accuracy_score(df["label"], df["ml_pred"])

print("Zero-shot Accuracy:", zero_acc)
print("ML Model Accuracy:", ml_acc)

Zero-shot Accuracy: 0.5833333333333334
ML Model Accuracy: 1.0


In [17]:
def top_k_accuracy(df, k=3):
    correct = 0
    for _, row in df.iterrows():
        if row["label"] in row["zero_shot_tags"][:k]:
            correct += 1
    return correct / len(df)

print("Top-3 Accuracy:", top_k_accuracy(df, 3))

Top-3 Accuracy: 1.0


In [18]:
df[["ticket", "label", "zero_top1", "few_shot_pred", "ml_pred"]]

,ticket,label,zero_top1,few_shot_pred,ml_pred
0,Internet is slow,Network Issue,Technical Issue,Classify the support ticket into one category:...,Network Issue
1,WiFi disconnects frequently,Network Issue,Network Issue,Classify the support ticket into one category:...,Network Issue
2,Server not responding,Network Issue,Technical Issue,Classify the support ticket into one category:...,Network Issue
3,Cannot login to account,Authentication,Technical Issue,Classify the support ticket into one category:...,Authentication
4,Forgot my password,Authentication,Authentication,Classify the support ticket into one category:...,Authentication
5,Login page not working,Authentication,Technical Issue,Classify the support ticket into one category:...,Authentication
6,Payment failed,Billing,Billing,Classify the support ticket into one category:...,Billing
7,Charged twice,Billing,Billing,Classify the support ticket into one category:...,Billing
8,Refund not received,Billing,Technical Issue,Classify the support ticket into one category:...,Billing
9,App crashes frequently,Technical Issue,Technical Issue,Classify the support ticket into one category:...,Technical Issue


In [32]:
def predict(ticket):
    zero = zero_shot_predict(ticket)
    ml = ml_predict(ticket)
    few = few_shot_predict(ticket)

    return {
        "Zero-shot Top 3": zero,
        "ML Prediction": ml,
        "Few-shot Prediction": few
    }

with gr.Blocks(title="Support Request Classifier") as demo:
    gr.Markdown("# Support Request Classifier\nZero-shot, Few-shot, and ML Model Predictions")
    with gr.Column():
        ticket_input = gr.Textbox(label="Ticket", lines=2, placeholder="Enter support request...")
        output_json = gr.JSON(label="Prediction Results")
        ticket_input.change(fn=predict, inputs=ticket_input, outputs=output_json)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://39208b8fa70487156c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
